# 04 — The Quantum Coupling SDP

Build the object the taxonomy named: a *quantum coupling* of two states, optimised by a semidefinite program — the operator-level lift of the Kantorovich LP.

**Prerequisites:** `04/03_trevisan_taxonomy`, `03/03_the_kantorovich_lp`.

**What you'll be able to do**
- Define a **quantum coupling** $\rho_{AB}$ of $\rho_A, \rho_B$: a bipartite density matrix whose partial traces *are* the two marginals.
- Read the **quantum OT semidefinite program** $\min_{\rho_{AB}\succeq 0}\ \mathrm{tr}(C\,\rho_{AB})$ as the term-by-term lift of the classical LP.
- Solve it in a few lines of cvxpy with `quantum_ot_sdp`, and run the first sanity check: identical states cost zero.
- Validate the solver against a closed form — for pure states the SWAP cost returns $1 - |\langle\psi|\phi\rangle|^2$.

In `04/03` you read Trevisan's map and committed this course to the **coupling SDP** — the entry sitting closest to the Kantorovich linear program of `03/03`. So far that commitment is a row in a table. This notebook makes it a computation you can run: we lift the classical coupling to an operator, lift the LP to an SDP, solve it, and check the answer against cases we already know cold. By the end you will have solved an honest operator-level transport problem and confirmed it against a closed form — the discipline that lets us trust every number the solver returns from here on.

In [ ]:
import numpy as np

from qot_course import viz
from qot_course.quantum.density import density_matrix
from qot_course.quantum_ot.sdp import (
    swap_cost,
    quantum_ot_sdp,
    quantum_wasserstein_squared_swap,
)

np.random.seed(42)
viz.use_course_style()

## 1. From the Kantorovich LP to an operator SDP

In `03/03` a transport plan was a **coupling**: a non-negative matrix $P$ whose row sums returned the source masses $a$ and whose column sums returned the target masses $b$. Optimal transport was then the linear program $\min_{P\ge 0}\ \langle C, P\rangle$ subject to those two marginal constraints. Everything we need to go quantum is already in that sentence — we replace each classical object by its operator analogue, one term at a time.

A classical distribution becomes a **density matrix**. A joint distribution over two systems becomes a **bipartite density matrix** $\rho_{AB}$ on $\mathcal{H}_A \otimes \mathcal{H}_B$. The classical marginals — summing the joint over one index — become **partial traces**, the operation that traces out one subsystem. So a *quantum coupling* of $\rho_A$ and $\rho_B$ is a bipartite density matrix whose partial traces are exactly those two states:

$$ \mathrm{tr}_B(\rho_{AB}) = \rho_A, \qquad \mathrm{tr}_A(\rho_{AB}) = \rho_B, \qquad \rho_{AB} \succeq 0. $$

The one substitution that carries all the quantum content is in the last constraint. Classically $P \ge 0$ means every *entry* is non-negative — the plan lives in the non-negative orthant. Quantum-mechanically $\rho_{AB} \succeq 0$ means the *operator* is positive semidefinite — its eigenvalues are non-negative. The PSD cone replaces the orthant, and that single change is what lets the coupling carry coherence a classical plan cannot express.

The objective lifts the same way: the classical dot product $\langle C, P\rangle = \sum_{ij} C_{ij} P_{ij}$ becomes the operator trace $\mathrm{tr}(C\,\rho_{AB})$ against a Hermitian **cost operator** $C$. Reading the dictionary term by term:

| Classical Kantorovich LP (`03/03`) | Quantum coupling SDP (here) |
|---|---|
| transport plan $P \in \mathbb{R}_{\ge 0}^{n \times m}$ | bipartite density matrix $\rho_{AB}$ on $\mathcal{H}_A \otimes \mathcal{H}_B$ |
| marginals $P\,\mathbf{1} = a,\ P^\top\mathbf{1} = b$ | partial traces $\mathrm{tr}_B(\rho_{AB}) = \rho_A,\ \mathrm{tr}_A(\rho_{AB}) = \rho_B$ |
| cost dot product $\langle C, P\rangle$ | operator trace $\mathrm{tr}(C\,\rho_{AB})$ |
| non-negativity $P \ge 0$ (the orthant) | positivity $\rho_{AB} \succeq 0$ (the PSD cone) |
| **linear** program | **semidefinite** program |

Putting the pieces together gives the **quantum optimal-transport SDP** (De Palma & Trevisan, 2021; Cole, Lostaglio, Verma & Wilde, 2023):

$$ \mathrm{QOT}(\rho_A, \rho_B) \,=\, \min_{\rho_{AB} \succeq 0}\ \mathrm{tr}(C\,\rho_{AB}) \quad \text{s.t.} \quad \mathrm{tr}_B(\rho_{AB}) = \rho_A,\ \ \mathrm{tr}_A(\rho_{AB}) = \rho_B. $$

It is a linear objective on a Hermitian PSD variable with linear partial-trace constraints — the textbook shape of a semidefinite program, and exactly the generalisation of the LP that the `04/03` taxonomy placed at the head of the coupling path.

## 2. The SDP in a few lines of cvxpy

A semidefinite program with a PSD-cone constraint and linear equality constraints is precisely what modern convex-optimisation libraries are built to solve. `qot_course.quantum_ot.sdp.quantum_ot_sdp` wraps the whole problem; its core is the direct transcription of the boxed program above:

```python
plan = cp.Variable((d_a * d_b, d_a * d_b), hermitian=True)
constraints = [
    plan >> 0,                                              # the PSD cone (rho_AB >= 0)
    cp.partial_trace(plan, (d_a, d_b), axis=1) == rho_a,    # tr_B(rho_AB) = rho_A
    cp.partial_trace(plan, (d_a, d_b), axis=0) == rho_b,    # tr_A(rho_AB) = rho_B
]
problem = cp.Problem(cp.Minimize(cp.real(cp.trace(cost @ plan))), constraints)
problem.solve(solver="CLARABEL")
```

Read it against the dictionary in §1 and every line lands on a row: the Hermitian variable *is* the quantum coupling, `plan >> 0` *is* the PSD cone replacing the orthant, the two `cp.partial_trace` lines *are* the marginal constraints, and `cp.trace(cost @ plan)` *is* the operator objective $\mathrm{tr}(C\,\rho_{AB})$. CLARABEL is the interior-point SDP solver shipped with cvxpy; the helper tightens its tolerances so qubit-scale problems return to roughly $10^{-9}$ accuracy, which is what lets us validate against closed forms below.

The cost operator $C$ is a modelling choice — it encodes what "far apart" means for these states, exactly as the ground-cost matrix did classically. This brick uses the **SWAP cost** $C = I - \mathrm{SWAP}$ of Cole et al. (2023): the SWAP operator exchanges the two subsystems, so $I - \mathrm{SWAP}$ charges nothing when the systems already agree and a positive amount when they differ. Its payoff is the clean closed form we validate in §4. (A second natural cost, the quadratic position operator that recovers the classical $W_2^2$ on commuting states, waits for the `04/05` synthesis.)

`quantum_ot_sdp(rho_a, rho_b, cost)` returns a pair: the optimal **value** and the optimal **plan** $\rho_{AB}^\star$ itself. We use only the value in this brick — reading the structure of the optimal plan is the work of `04/05`.

## 3. Sanity check — identical states cost zero

Before trusting any optimiser, give it a problem whose answer you already know. The simplest is transporting a state onto *itself*: $\rho_A = \rho_B = \rho$. There is nothing to move, so the transport cost must be zero.

The SDP confirms it has a feasible zero-cost coupling on hand. Write $\rho = \sum_k \lambda_k |k\rangle\langle k|$ in its eigenbasis and form the perfectly correlated coupling $\rho_{AB} = \sum_k \lambda_k\, |k\rangle\langle k| \otimes |k\rangle\langle k|$. Its partial traces are both $\rho$ (so it is feasible), and it places all its weight on states $|k\rangle|k\rangle$ where the two subsystems agree — exactly where the SWAP cost $I - \mathrm{SWAP}$ vanishes. The minimum can only be zero. We run the SDP on $\rho = |+\rangle\langle+|$ with the qubit SWAP cost, the call the contract names: `quantum_ot_sdp(rho, rho, swap_cost(2))`.

In [ ]:
rho = density_matrix([1.0, 1.0])  # |+> = (|0> + |1>)/sqrt(2), pre-normalised by density_matrix
value_identical, _ = quantum_ot_sdp(rho, rho, swap_cost(2))

# The SDP solves to interior-point tolerance, so the value carries ~1e-9 solver
# noise (and may land slightly below zero). Format it; do not read the trailing digits.
print(f"QOT(|+><+|, |+><+|) with SWAP cost = {value_identical:.2e}")
print(f"  |value| within solver tolerance of 0?  {abs(value_identical) < 1e-6}")

**Read the output.** The value is zero up to the solver's interior-point tolerance — a number on the order of $10^{-9}$, which may sit a hair below zero rather than exactly on it. That tiny residual is numerical, not physical: it is the gap CLARABEL leaves when it stops iterating, and the right way to read it is "zero to tolerance", which is why we format it with `.2e` and check $|\text{value}| < 10^{-6}$ instead of testing equality with $0$. Physically the message is clean: identical states admit a perfect coupling, so there is no transport to pay for. The solver passes the first test we can check by hand.

## 4. Pure-state closed form — validating against a known answer

The zero-cost check confirms the solver behaves on the trivial case; now we hold it against a *non-trivial* answer we can compute independently. For two **pure** states $\rho_A = |\psi\rangle\langle\psi|$ and $\rho_B = |\phi\rangle\langle\phi|$ there is a closed form. Pure marginals leave no slack — the only coupling with those partial traces is the product $\rho_{AB} = |\psi\rangle\langle\psi| \otimes |\phi\rangle\langle\phi|$ — so the SDP has a single feasible point and the minimum is its cost. Evaluating the SWAP cost on that product gives

$$ \mathrm{tr}\!\bigl((I - \mathrm{SWAP})\,|\psi\rangle\langle\psi| \otimes |\phi\rangle\langle\phi|\bigr) \,=\, 1 - |\langle\psi|\phi\rangle|^2. $$

The transport cost between two pure states is one minus their squared overlap: identical states ($|\langle\psi|\phi\rangle|^2 = 1$) cost zero, orthogonal states ($|\langle\psi|\phi\rangle|^2 = 0$) cost the maximum of one. We draw five random qubit pairs and, for each, compare the SDP value (via the convenience wrapper `quantum_wasserstein_squared_swap`, which calls the SDP with the SWAP cost) against this formula.

In [ ]:
rng = np.random.default_rng(7)


def random_qubit(rng: np.random.Generator) -> np.ndarray:
    """Draw a Haar-ish random pure qubit: complex Gaussian entries, normalised."""
    psi = rng.normal(size=2) + 1j * rng.normal(size=2)
    return psi / np.linalg.norm(psi)


header = f"{'|<psi|phi>|^2':>14s}  {'closed form':>12s}  {'SDP value':>11s}  {'|difference|':>13s}"
print(header)
print("-" * len(header))
for _ in range(5):
    psi = random_qubit(rng)
    phi = random_qubit(rng)
    overlap_sq = abs(np.vdot(psi, phi)) ** 2
    closed_form = 1.0 - overlap_sq
    sdp_value = quantum_wasserstein_squared_swap(
        density_matrix(psi), density_matrix(phi)
    )
    # Round the displayed columns to 4 decimals; show the residual in scientific
    # notation so the ~1e-9..1e-5 solver gap is legible rather than misread as a value.
    print(
        f"{overlap_sq:>14.4f}  {closed_form:>12.4f}  "
        f"{sdp_value:>11.4f}  {abs(sdp_value - closed_form):>13.1e}"
    )

**Read the table.** Each row is one random qubit pair. The squared overlaps scatter across the unit interval — from a near-aligned pair around $0.97$ to a well-separated one near $0.44$ — and the **closed form** column is $1$ minus that overlap. The decisive comparison is the next two columns: the **SDP value** matches the closed form to four decimals on every row, with the **|difference|** column reporting a residual between roughly $10^{-9}$ and $10^{-5}$ — the interior-point tolerance, never a real disagreement. This is validation in both directions at once. The closed form vouches for the solver, and the solver vouches for the closed form, so when we turn the SDP loose on states that *have* no closed form — mixed states, larger systems — we already have grounds to trust the number it returns.

## Your turn

**Warm-up.** Predict, then check, the cost between two states you build by hand. Take $|0\rangle$ and $|1\rangle$ — orthogonal, so $|\langle 0 | 1\rangle|^2 = 0$. State the value the closed form of §4 demands *before* solving, then call `quantum_wasserstein_squared_swap(density_matrix([1, 0]), density_matrix([0, 1]))` and confirm it lands on that number to solver tolerance. What is the largest transport cost the SWAP problem can ever return on a qubit pair, and which geometric relationship between the two states produces it?

**Go further.** A returned plan should be an honest coupling. Call `quantum_ot_sdp(rho_a, rho_b, swap_cost(2))` for two states of your choice and keep the *second* return value, the optimal plan $\rho_{AB}^\star$. Verify the two marginal constraints directly: trace out each subsystem and check the results equal $\rho_A$ and $\rho_B$. (The partial-trace helper lives in `qot_course.quantum.composite`; recall from §1 that tracing out system $B$ must return $\rho_A$, and tracing out $A$ must return $\rho_B$.)

**Challenge.** Reason about the cost operator as a modelling choice. The SWAP cost charges by disagreement between subsystems; describe in words what kind of quantum coupling — what alignment between the $A$ and $B$ registers — would drive $\mathrm{tr}(C\,\rho_{AB})$ toward its minimum, and what would push it toward its maximum. Then predict how the optimal value should change if you replaced $C = I - \mathrm{SWAP}$ with $C = 2(I - \mathrm{SWAP})$, and argue your prediction from the linearity of the objective alone (no solving required).

## What you built

- You defined a **quantum coupling** $\rho_{AB}$ — a bipartite density matrix whose partial traces *are* the two marginals — and saw it as the operator lift of the classical coupling from `03/03`, with the PSD cone standing in for the non-negative orthant.
- You read the **quantum OT semidefinite program** $\min_{\rho_{AB}\succeq 0}\ \mathrm{tr}(C\,\rho_{AB})$ as the term-by-term generalisation of the Kantorovich LP, and saw it transcribe into a handful of lines of cvxpy.
- You **solved** it with `quantum_ot_sdp` and ran the first sanity check: identical states cost zero to solver tolerance.
- You **validated** the solver against a closed form — for pure states the SWAP cost returns $1 - |\langle\psi|\phi\rangle|^2$, matched on five random qubit pairs.

Take a moment with what you have done: you set up and solved an optimal transport problem *between quantum states*, then proved the answer correct against a formula you derived by hand. That is the coupling path of `04/03` turned from a promise into a working tool — and the validation discipline that makes every later result trustworthy.

## What's next

In `04/05_solving_qot_in_cvxpy` this machinery delivers the punchline the module has been building toward: the SDP separates $|+\rangle\langle+|$ from $I/2$, the two states classical OT could not tell apart, and we read the optimal coupling and close the quantum-OT row of the running dictionary.

## References

- G. De Palma & D. Trevisan, "Quantum optimal transport with quantum channels", *Annales Henri Poincaré* 22, 3199–3234 (2021). DOI:10.1007/s00023-021-01042-3
- S. Cole, M. Lostaglio, K. Verma & M. M. Wilde, "Quantum Wasserstein distance based on an optimization over separable states", *IEEE Transactions on Information Theory* (2023). DOI:10.1109/TIT.2023.3328953
- D. Trevisan, "Optimal transport methods for quantum systems", arXiv:2202.02091 (2022). DOI:10.48550/arXiv.2202.02091
- S. P. Boyd & L. Vandenberghe, *Convex Optimization*, Cambridge University Press (2004), chs. 4 & 11 (semidefinite programming).

**Previous:** `notebooks/04_QuantumOptimalTransport/03_trevisan_taxonomy.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/05_solving_qot_in_cvxpy.ipynb`